# Stage 2 — Custom TF-IDF Vectorizer (no sklearn)

Builds a from-scratch TF-IDF vectorizer combining:
- **Word unigrams and bigrams** (whitespace-split — all 23 languages here are written with spaces between words in modern digitized text)
- **Character 2-, 3-, and 4-grams** (sliding window over the raw sentence string, spaces included)

Each gram is tagged with its type (`W1|`, `W2|`, `C2|`, `C3|`, `C4|`) before being added to one shared vocabulary, so e.g. the word unigram "इस" and a character 2-gram "इस" occupy different feature dimensions.

**Design choices, made explicit:**
- **IDF formula (smoothed):** `idf(t) = ln((1+N)/(1+df(t))) + 1` — the `+1`s inside avoid division/log-of-zero, and the trailing `+1` keeps idf strictly positive even for a term that appears in every document (this is the same smoothing convention scikit-learn's `TfidfVectorizer` uses by default, so it's a well-established choice, not an ad hoc one).
- **Vocabulary from TRAIN only**, frozen before touching val/test — avoids data leakage.
- **`min_df=3`** (a gram must appear in at least 3 distinct training sentences to become a feature). Without pruning, the raw vocabulary is ~944k grams, of which the overwhelming majority are singletons — a gram seen in only 1-2 training sentences gives a linear model essentially nothing to generalize from. `min_df=3` cuts this to ~215k features, which is both more statistically meaningful and far cheaper to train on.
- **L2-normalize each document's TF-IDF vector** — without this, longer sentences would have systematically larger vector norms than shorter ones for reasons that have nothing to do with language identity, which would bias a linear classifier. This is standard practice (again matching sklearn's default behavior).
- **Sparse storage (`scipy.sparse.csr_matrix`)** throughout — a dense 18,400 × 215,000 float64 matrix would be ~31 GB; the actual sparse matrix is under 50 MB, since each sentence only activates ~200 of the 215k dimensions.


In [1]:
import time
import json
import os
from collections import Counter

import numpy as np
import pandas as pd
import scipy.sparse as sp

DATA_DIR = "../data"
MIN_DF = 3

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
print(train_df.shape, val_df.shape, test_df.shape)


(18400, 2) (2300, 2) (2300, 2)


In [2]:
def tokenize(text):
    """Extract tagged word 1/2-grams and char 2/3/4-grams from one sentence."""
    text = text.strip()
    words = text.split()
    grams = []

    for w in words:
        grams.append("W1|" + w)
    for i in range(len(words) - 1):
        grams.append("W2|" + words[i] + " " + words[i + 1])

    for n in (2, 3, 4):
        for i in range(len(text) - n + 1):
            grams.append(f"C{n}|" + text[i:i + n])

    return grams


# quick look at what tokenize() produces for one sentence
example = train_df["text"].iloc[0]
example_grams = tokenize(example)
print("sentence:", example)
print("total grams:", len(example_grams))
print("sample:", example_grams[:6], "...", example_grams[-4:])


sentence: গুৱাহাটী, ২৫ এপ্ৰিলঃ সমগ্ৰ দেশতে আজি ২৪ সংখ্যক তথা অন্তিমগৰাকী জৈন তীর্থাংকৰ ভগৱান মহাবীৰৰ জন্ম জয়ন্তী পালন কৰা হৈছে।
total grams: 385
sample: ['W1|গুৱাহাটী,', 'W1|২৫', 'W1|এপ্ৰিলঃ', 'W1|সমগ্ৰ', 'W1|দেশতে', 'W1|আজি'] ... ['C4|া হৈ', 'C4| হৈছ', 'C4|হৈছে', 'C4|ৈছে।']


In [3]:
class NgramTfidfVectorizer:
    """TF-IDF vectorizer over combined word/char n-grams. No sklearn used anywhere."""

    def __init__(self, min_df=3):
        self.min_df = min_df
        self.vocab_ = None   # gram -> column index
        self.idf_ = None     # np.ndarray, aligned to vocab_ indices

    def fit(self, texts):
        df_counter = Counter()
        for text in texts:
            df_counter.update(set(tokenize(text)))  # count each gram once per doc

        kept = sorted(g for g, c in df_counter.items() if c >= self.min_df)
        self.vocab_ = {g: i for i, g in enumerate(kept)}

        n_docs = len(texts)
        df_arr = np.array([df_counter[g] for g in kept], dtype=np.float64)
        self.idf_ = np.log((1.0 + n_docs) / (1.0 + df_arr)) + 1.0
        return self

    def transform(self, texts):
        rows, cols, data = [], [], []
        for row_idx, text in enumerate(texts):
            tf = Counter(tokenize(text))
            vec = {}
            for gram, count in tf.items():
                j = self.vocab_.get(gram)
                if j is not None:  # unseen grams (not in train vocab) are dropped
                    vec[j] = count * self.idf_[j]

            norm = np.sqrt(sum(v * v for v in vec.values())) or 1.0
            for j, v in vec.items():
                rows.append(row_idx)
                cols.append(j)
                data.append(v / norm)

        return sp.csr_matrix((data, (rows, cols)), shape=(len(texts), len(self.vocab_)))

    def fit_transform(self, texts):
        return self.fit(texts).transform(texts)


In [4]:
t0 = time.time()
vectorizer = NgramTfidfVectorizer(min_df=MIN_DF)
X_train = vectorizer.fit_transform(train_df["text"].tolist())
t1 = time.time()
print(f"fit_transform(train) in {t1 - t0:.1f}s -> vocab size = {len(vectorizer.vocab_):,}")
print(f"X_train shape={X_train.shape}  nnz={X_train.nnz:,}  avg nonzeros/row={X_train.nnz / X_train.shape[0]:.1f}")


fit_transform(train) in 5.8s -> vocab size = 214,895
X_train shape=(18400, 214895)  nnz=3,690,535  avg nonzeros/row=200.6


In [5]:
t0 = time.time()
X_val = vectorizer.transform(val_df["text"].tolist())
X_test = vectorizer.transform(test_df["text"].tolist())
t1 = time.time()
print(f"transform(val, test) in {t1 - t0:.1f}s")
print("X_val: ", X_val.shape, " nnz=", X_val.nnz)
print("X_test:", X_test.shape, " nnz=", X_test.nnz)


transform(val, test) in 0.9s
X_val:  (2300, 214895)  nnz= 424917
X_test: (2300, 214895)  nnz= 435669


## Sanity checks

1. Feature-type breakdown of the vocabulary — confirms all 5 gram types survived pruning, not just one type dominating.
2. Out-of-vocabulary rate on val/test — how many grams in unseen sentences simply don't exist in the train-derived vocabulary (expected to be nonzero, especially for word bigrams; should not be so high that vectors become mostly empty).
3. Top TF-IDF terms for a couple of example sentences — should be recognizable, language-distinctive tokens, not noise.


In [6]:
from collections import Counter as _Counter
type_counts = _Counter(g.split("|", 1)[0] for g in vectorizer.vocab_)
for t in ["W1", "W2", "C2", "C3", "C4"]:
    print(f"{t}: {type_counts[t]:,}")


W1: 13,766
W2: 4,038
C2: 17,647
C3: 68,332
C4: 111,112


In [7]:
def oov_rate(texts, vectorizer, sample_size=500):
    seen, kept = 0, 0
    for text in texts[:sample_size]:
        grams = tokenize(text)
        seen += len(grams)
        kept += sum(1 for g in grams if g in vectorizer.vocab_)
    return 1 - kept / seen

print(f"val OOV gram rate:  {oov_rate(val_df['text'].tolist(), vectorizer):.1%}")
print(f"test OOV gram rate: {oov_rate(test_df['text'].tolist(), vectorizer):.1%}")


val OOV gram rate:  18.3%
test OOV gram rate: 18.4%


In [8]:
inv_vocab = {i: g for g, i in vectorizer.vocab_.items()}

for row_idx in [0, 4500, 9000]:
    row = X_train.getrow(row_idx)
    top = sorted(zip(row.indices, row.data), key=lambda p: -p[1])[:8]
    label = train_df["label"].iloc[row_idx]
    text = train_df["text"].iloc[row_idx]
    print(f"[{label}] {text}")
    print("  top grams:", [(inv_vocab[j], round(v, 3)) for j, v in top])
    print()


[asm_Beng] গুৱাহাটী, ২৫ এপ্ৰিলঃ সমগ্ৰ দেশতে আজি ২৪ সংখ্যক তথা অন্তিমগৰাকী জৈন তীর্থাংকৰ ভগৱান মহাবীৰৰ জন্ম জয়ন্তী পালন কৰা হৈছে।
  top grams: [('C2|মগ', np.float64(0.118)), ('C2| জ', np.float64(0.112)), ('C2|ন্', np.float64(0.101)), ('C2|ৰ ', np.float64(0.1)), ('C2|ন ', np.float64(0.099)), ('C2| ২', np.float64(0.096)), ('C2|তী', np.float64(0.094)), ('C2|ৱা', np.float64(0.084))]

[guj_Gujr] હવામાન વિભાગે આગામી બે અઠવાડિયા સુધી વરસાદ નહીં વરશે તેવા સંકેત આપ્યા છે.
  top grams: [('C2| વ', np.float64(0.136)), ('C2|વા', np.float64(0.132)), ('C3| વર', np.float64(0.128)), ('C2|ા ', np.float64(0.123)), ('C2|ે ', np.float64(0.12)), ('C2|વર', np.float64(0.116)), ('C3|ા સ', np.float64(0.112)), ('C3|યા ', np.float64(0.109))]

[mar_Deva] त्यानंतर मुंबईत एकूण ३ प्रचारसभा होणार असल्याचे सांगण्यात येते.
  top grams: [('C3|्या', np.float64(0.12)), ('C4|चारस', np.float64(0.111)), ('C4|ारसभ', np.float64(0.111)), ('W1|येते.', np.float64(0.109)), ('C3|एकू', np.float64(0.106)), ('C3|कूण', np.float64(0.106)

## Save vectorizer artifacts + transformed matrices for Stage 3

In [9]:
sp.save_npz(os.path.join(DATA_DIR, "X_train.npz"), X_train)
sp.save_npz(os.path.join(DATA_DIR, "X_val.npz"), X_val)
sp.save_npz(os.path.join(DATA_DIR, "X_test.npz"), X_test)

train_df[["label"]].to_csv(os.path.join(DATA_DIR, "y_train.csv"), index=False)
val_df[["label"]].to_csv(os.path.join(DATA_DIR, "y_val.csv"), index=False)
test_df[["label"]].to_csv(os.path.join(DATA_DIR, "y_test.csv"), index=False)

with open(os.path.join(DATA_DIR, "vocab.json"), "w", encoding="utf-8") as f:
    json.dump(vectorizer.vocab_, f, ensure_ascii=False)
np.save(os.path.join(DATA_DIR, "idf.npy"), vectorizer.idf_)

print("Saved X_{train,val,test}.npz, y_{train,val,test}.csv, vocab.json, idf.npy to", os.path.abspath(DATA_DIR))


Saved X_{train,val,test}.npz, y_{train,val,test}.csv, vocab.json, idf.npy to /home/deeraj/LLM/Lab1/data
